# Scaling Physical AI & Robotics Systems with Ray

## TLDR

This is the overview for a six-notebook course on building an **end-to-end
physical-AI workflow** with **Ray on Anyscale**. Across the series you stream
robotics data, fine-tune a Vision-Language-Action (VLA) policy, serve it and
evaluate it in simulation, close the training loop with sim data, pre-train a
**world model** at scale, and **distill** a large model into a small backbone
fit for a robot. The models are the workload; the lesson is the infrastructure —
**Ray Data, Ray Train, Ray Serve, and Ray remote tasks** — that makes all of it
run on one cluster.

> **Scale up to learn; scale down to deploy.**

## Why the infrastructure is the hard part

Modern robotics teams improve policies with a data flywheel: deploy, observe
failures, collect targeted data, retrain. It works, but closing that loop is a
**distributed-systems problem**, not just a modeling one. Four workloads have to
share a cluster and pass state between each other:

1. **Streaming training data at scale.** Robotics datasets are hundreds of
   thousands of multi-camera video frames. You cannot snapshot them onto every
   node — you need a streaming pipeline that works the same at 10 GB or 10 TB.
2. **Distributed training at the scale of the model.** PI0.5 is 3.4B
   parameters; a world model can be larger. Multi-GPU DDP, gradient
   aggregation, fault-tolerant checkpointing — with no hand-written boilerplate.
3. **Closed-loop simulation evaluation.** The policy and the simulator can't
   share a GPU or a process; they run on separate machines and talk over the
   network, step by step.
4. **Routing data back into training.** Sim trajectories must reach the next
   training round; checkpoints must move from trainers to serving replicas. The
   shared filesystem and dataset union have to be first-class, not glue scripts.

Most teams that have the models don't have this infrastructure. Ray on Anyscale
provides it as a handful of composable primitives.

## The lifecycle this course builds

```
   ┌─────────┐   ┌──────────────┐   ┌────────────┐   ┌─────────────┐
   │  DATA   │──▶│  FINE-TUNE   │──▶│   SERVE    │──▶│   SIMULATE  │
   │ Ray Data│   │  a VLA       │   │ the policy │   │  & evaluate │
   │ stream  │   │  Ray Train   │   │ Ray Serve  │   │ Ray tasks   │
   └─────────┘   └──────────────┘   └────────────┘   └──────┬──────┘
        ▲                                                   │
        │                  CLOSE THE LOOP                   │
        └───────────  filter by reward, union  ◀────────────┘
                              │
            ┌─────────────────┴───────────────────┐
            ▼                                      ▼
   ┌──────────────────┐                  ┌────────────────────┐
   │  WORLD MODEL     │                  │   DISTILL FOR EDGE │
   │  pre-train at    │  ───────────▶    │  big teacher →     │
   │  scale (V-JEPA)  │   scale down     │  small student     │
   │  Ray Train       │                  │  Ray Train         │
   └──────────────────┘                  └────────────────────┘
```

The arc runs from **using a simulator as a data factory** to **learning the
simulator itself** (a world model), and finally to **shrinking models down**
so they fit on a robot. The same Ray Train code carries you from a 3.4B model on
many GPUs to a MobileNet that runs on the edge.

## What you will learn

By the end of the series you will be able to:

- **Stream** large multi-camera robotics datasets with **Ray Data** — no local
  copy on any node — and preprocess them on a CPU pool that scales independently
  of your GPUs.
- **Fine-tune and pre-train** large robotics models with **Ray Train** — DDP,
  mixed precision, fault-tolerant checkpointing — scaling from a few GPUs to
  hundreds by changing one config value.
- **Serve** a policy as an HTTP service with **Ray Serve**, the same primitive
  used to serve LLMs in production.
- **Fan out** parallel simulation rollouts as **Ray remote tasks**, each on its
  own GPU, fully fault-isolated.
- **Close the loop**: filter sim trajectories by reward and `union()` them back
  into the training stream in one line.
- **Distill** a large teacher model into a small, deployable student backbone.

## The through-line: one Ray surface, taught once, reused everywhere

The whole course rests on a small, repeating set of Ray APIs. Learn them in
01–02 and you will recognize them unchanged in every later notebook:

| Notebook | Ray primitive(s) | What it does | Outline bullet |
|----------|------------------|--------------|----------------|
| 01 Data pipelines | `ray.data` / `read_lerobot` / `map_batches` | Stream + preprocess LeRobot v3 video | Robotics data prep |
| 02 VLA fine-tuning | `TorchTrainer`, `prepare_model`, `train.report`, `ScalingConfig` | DDP fine-tune PI0.5 | VLA fine-tuning |
| 03 Serve + sim eval | `@serve.deployment`, `serve.run`, `@ray.remote(num_gpus=1)` | Policy server + Isaac Lab fan-out + close the loop | Distributed sim & eval; scalable inference |
| 04 World model | `read_lerobot`, `TorchTrainer` (×2 phases) | Pre-train V-JEPA + online adaptation | World-model pre-training at scale; VLA pre-training |
| 05 Distillation | `ray.data`, `TorchTrainer` | Teacher→student for edge deploy | Scalable inference (edge) |

The same `TorchTrainer` + `prepare_model` + `train.report` + `FailureConfig` +
`ScalingConfig` drives fine-tuning (02), world-model pre-training (04), and
distillation (05). The same `lerobot_datasource` feeds 01–04. **Change the
config, not the code.**

## How this scales on Anyscale

Every notebook runs at deliberately small "smoke" scale so it finishes in
minutes on a 4× L4 cluster. The same code path scales on Anyscale by adjusting
configuration, not rewriting logic:

| Lever | This course | Production |
|-------|-------------|------------|
| Dataset | streamed subset | full corpus (10 TB+), same `hf://` / S3 path |
| Train workers | 4 × L4 | 8–64+ × A100/H100 — change `ScalingConfig(num_workers=N)` |
| Train steps | ~50–200 (smoke) | full epochs |
| Sim workers | 2 | as many GPUs as you have — change `SIM_WORKERS` |
| Serve replicas | 1 | autoscaled behind a load balancer |

Anyscale manages cluster scaling, GPU scheduling, and shared storage; you change
a few numbers.

## Course map — how to navigate

Read the notebooks in order; each builds on the last and cross-references it.
All ship **pre-run** (outputs committed), so you can read the full story without
a cluster, then re-run any notebook yourself.

| # | Notebook | Focus |
|---|----------|-------|
| 00 | this notebook | The lifecycle and the through-line |
| 01 | `01_robotics_data_pipelines.ipynb` | Stream + preprocess robotics video (Ray Data) |
| 02 | `02_vla_finetuning.ipynb` | Distributed DDP fine-tune of PI0.5 (Ray Train) |
| 03 | `03_serving_and_sim_eval.ipynb` | Serve + Isaac Lab fan-out + close the loop |
| 04 | `04_world_model_pretraining.ipynb` | V-JEPA world model pre-training at scale |
| 05 | `05_distillation_for_edge.ipynb` | Distill a teacher into an edge-ready student |

**Prerequisites:** a 4× L4 Anyscale cluster on the course image, and (for 02/03)
`export HF_TOKEN=...` with access to the gated `google/paligemma-3b-pt-224`. See
`README.md` for the full cluster image and tested versions.

**A note on scope:** the VLA half (02/03) runs PI0.5 — fine-tuned on LIBERO's
Panda — on Isaac Lab's Franka. The embodiments differ, so expect exploratory
motion, not task success. We validate the orchestration loop, not manipulation.

## Cell 1: Confirm your cluster

**What you do** — connect to the Ray cluster and print its resources.

**What to check** — you should see **4 GPUs** and a pool of CPUs. On Anyscale,
`ray.init(address="auto")` attaches to the managed cluster that is already
running.

**Why it matters** — this single line is all the infrastructure setup the rest
of the course needs. Everything downstream (Ray Data preprocessing, Ray Train
DDP, Ray Serve, sim tasks) draws from these resources.

In [1]:
import ray

try:
    ray.init(address="auto", ignore_reinit_error=True)
except ConnectionError:
    ray.init(ignore_reinit_error=True)

res = ray.cluster_resources()
print(f"GPUs:   {int(res.get('GPU', 0))}")
print(f"CPUs:   {int(res.get('CPU', 0))}")
print(f"Memory: {res.get('memory', 0) / 1e9:.0f} GiB")

2026-06-08 17:52:53,881	INFO worker.py:1821 -- Connecting to existing Ray cluster at address: 10.0.69.43:6379...


2026-06-08 17:52:53,894	INFO worker.py:1998 -- Connected to Ray cluster. View the dashboard at https://session-ih3rjmvr7i1pepvl4xqj5nplaq.i.anyscaleuserdata.com 


2026-06-08 17:52:53,908	INFO packaging.py:463 -- Pushing file package 'gcs://_ray_pkg_75617797bfc6eb94925121a766090258c62203e9.zip' (0.24MiB) to Ray cluster...


2026-06-08 17:52:53,910	INFO packaging.py:476 -- Successfully pushed file package 'gcs://_ray_pkg_75617797bfc6eb94925121a766090258c62203e9.zip'.


GPUs:   4
CPUs:   64
Memory: 309 GiB


/home/ray/anaconda3/lib/python3.11/site-packages/ray/_private/worker.py:2046: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


## Where to go next

Continue to **`01_robotics_data_pipelines.ipynb`**, where Ray Data streams the
LIBERO dataset straight from HuggingFace — hundreds of thousands of Franka
manipulation frames — without ever landing a copy on disk.